# Part 1: Foundations of Self-Attention

**Goal:** Build intuition for single-head self-attention from scratch using only NumPy.

---

## 1.1 The Task: Argmax Broadcast

We use a toy task to study attention:
- **Input:** A sequence of integers, e.g., `[3, 7, 2, 5]`
- **Target:** Every position outputs the maximum value, e.g., `[7, 7, 7, 7]`

This requires the model to:
1. **Attend** to the position containing the max
2. **Broadcast** that value to all positions

If attention works correctly, we should see the attention matrix concentrate weight on the argmax position.

---

## 1.2 Self-Attention Architecture

For a sequence of $n$ positions, each embedded as a $d$-dimensional vector:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

Where:
- $Q = XW_Q$ (queries)
- $K = XW_K$ (keys)
- $V = XW_V$ (values)
- $H = ZW_H$ (output projection)

**Key insight:** The softmax normalizes each row of $QK^T$ to sum to 1, creating a probability distribution over positions. Each position "attends" to other positions based on query-key similarity.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)

## 1.3 Data Generator

In [3]:
def generate_argmax_sample(dim=4, value_range=(0, 10), seed=None):
    """Toy task: input x is length-dim ints; target y repeats argmax(x)."""
    if seed is not None:
        np.random.seed(seed)
    x = np.random.randint(value_range[0], value_range[1], size=(dim,))
    max_val = np.max(x)
    y = np.full((dim,), max_val)
    return x, y


def make_dataset(num_samples=1000, dim=4, value_range=(0, 10), seed=0):
    """Simple fixed-length dataset (used for quick sanity checks)."""
    rng = np.random.default_rng(seed)
    dataset = []
    for _ in range(num_samples):
        x = rng.integers(value_range[0], value_range[1], size=(dim,))
        y = np.full((dim,), int(np.max(x)))
        dataset.append((x, y))
    return dataset


# Demo
x_demo, y_demo = generate_argmax_sample(dim=5, seed=0)
print(f"Input:  {x_demo}")
print(f"Target: {y_demo}")
print(f"\nThe model must learn to output {y_demo[0]} at every position.")

Input:  [5 0 3 3 7]
Target: [7 7 7 7 7]

The model must learn to output 7 at every position.


## 1.4 Core Functions

### Softmax

In [ ]:
def row_softmax(scores: np.ndarray) -> np.ndarray:
    """
    Row-wise softmax (numerically stable).
    Each row is normalized to sum to 1.
    """
    scores = scores - np.max(scores, axis=1, keepdims=True)  # numerical stability
    exp_scores = np.exp(scores)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

### Forward Pass

In [ ]:
def attention_forward(hidden_layer, W_Q, W_K, W_V, W_H, return_cache=False, softmax_fn=None):
    """
    Single-head self-attention + output projection.
    
    Args:
        hidden_layer: (n, d_model) input embeddings
        W_Q, W_K, W_V: (d_model, d_model) projection matrices
        W_H: (d_model, d_model) output projection
        return_cache: if True, return intermediate values for backprop
        softmax_fn: optional custom softmax (for experiments)
    
    Returns:
        H: (n, d_model) output embeddings
        cache: dict of intermediates (if return_cache=True)
    """
    if softmax_fn is None:
        softmax_fn = row_softmax
    
    d_k = W_Q.shape[1]
    
    # Project to Q, K, V
    Q = hidden_layer @ W_Q
    K = hidden_layer @ W_K
    V = hidden_layer @ W_V
    
    # Scaled dot-product attention
    scores = (Q @ K.T) / np.sqrt(d_k)
    A = softmax_fn(scores)  # attention weights
    Z = A @ V               # weighted sum of values
    H = Z @ W_H             # output projection
    
    if not return_cache:
        return H
    
    cache = {"Q": Q, "K": K, "V": V, "scores": scores, "A": A, "Z": Z, "H": H}
    return H, cache


def mse(pred, target):
    """Mean squared error."""
    diff = pred - target
    return np.mean(diff * diff)

### Backpropagation + SGD Step

We compute gradients manually for each weight matrix.

In [6]:
def attention_step(W_Q, W_K, W_V, W_H, hidden_layer, target, lr=1e-3, clip=1.0, softmax_fn=None):
    """
    One SGD step on a single sequence.
    
    Returns:
        Updated weights, loss, cache, gradient norms
    """
    pred, cache = attention_forward(hidden_layer, W_Q, W_K, W_V, W_H, 
                                     return_cache=True, softmax_fn=softmax_fn)
    loss = mse(pred, target)
    
    # Gradient of MSE: d(loss)/d(pred) = 2(pred - target) / n
    dH = (2.0 / pred.size) * (pred - target)
    
    # Backprop through output projection
    Z, A, V, Q, K = cache["Z"], cache["A"], cache["V"], cache["Q"], cache["K"]
    dW_H = Z.T @ dH
    dZ = dH @ W_H.T
    
    # Backprop through attention
    dV = A.T @ dZ
    dA = dZ @ V.T
    
    # Backprop through softmax: dS = A * (dA - sum(dA * A, keepdims))
    dS = A * (dA - np.sum(dA * A, axis=1, keepdims=True))
    
    d_k = W_Q.shape[1]
    dQ = (dS @ K) / np.sqrt(d_k)
    dK = (dS.T @ Q) / np.sqrt(d_k)
    
    dW_Q = hidden_layer.T @ dQ
    dW_K = hidden_layer.T @ dK
    dW_V = hidden_layer.T @ dV
    
    # Track gradient norms (for debugging)
    grad_norms = {
        "W_Q": float(np.linalg.norm(dW_Q)),
        "W_K": float(np.linalg.norm(dW_K)),
        "W_V": float(np.linalg.norm(dW_V)),
        "W_H": float(np.linalg.norm(dW_H)),
    }
    
    # Gradient clipping (prevents exploding gradients)
    dW_Q = np.clip(dW_Q, -clip, clip)
    dW_K = np.clip(dW_K, -clip, clip)
    dW_V = np.clip(dW_V, -clip, clip)
    dW_H = np.clip(dW_H, -clip, clip)
    
    # SGD update
    W_Q = W_Q - lr * dW_Q
    W_K = W_K - lr * dW_K
    W_V = W_V - lr * dW_V
    W_H = W_H - lr * dW_H
    
    return W_Q, W_K, W_V, W_H, loss, cache, grad_norms

### Helper Functions

In [7]:
def make_hidden_and_target(x_raw, y_raw, W_embed, scale=10.0):
    """
    Embed scalar inputs into d_model space.
    
    CRITICAL PARAMETER: scale
    - Divides raw values before embedding
    - scale=10.0 → inputs in [0,1) → tiny activations → tiny gradients
    - scale=1.0  → inputs in [0,10) → larger activations → meaningful gradients
    
    This is a key parameter that affects gradient flow!
    """
    x = (np.array(x_raw, dtype=np.float64).reshape(-1, 1)) / scale
    y = (np.array(y_raw, dtype=np.float64).reshape(-1, 1)) / scale
    hidden_layer = x @ W_embed
    target = y @ W_embed
    return hidden_layer, target


def decode_embedding_to_scalar(embedded, W_embed, scale=10.0):
    """Project embedded vectors back to scalar (least-squares)."""
    w = W_embed.reshape(-1)
    denom = float(w @ w) + 1e-12
    scalars = (embedded @ w) / denom
    return scalars * scale


def attention_row_entropy(A: np.ndarray) -> float:
    """
    Mean entropy across rows of attention matrix.
    
    - High entropy ≈ uniform attention (not learning structure)
    - Low entropy ≈ sharp attention (focusing on specific positions)
    """
    eps = 1e-12
    P = np.clip(A, eps, 1.0)
    H = -np.sum(P * np.log(P), axis=1)
    return float(np.mean(H))


def inspect_cache(cache, max_rows=8):
    """Print compact view of attention internals."""
    scores, A, H = cache["scores"], cache["A"], cache["H"]
    n = min(max_rows, scores.shape[0])
    print("scores (scaled dot-product):")
    print(scores[:n, :n])
    print("\nA (attention weights, rows sum to 1):")
    print(A[:n, :n])
    print(f"\nmean row entropy(A): {attention_row_entropy(A):.4f}")
    print("\nH (output, embedded):")
    print(H[:n])

---

## 1.6 Key Hyperparameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `scale` | 10.0 | Divides inputs before embedding |
| `lr` | 1e-3 | Learning rate (step size) |
| `clip` | 1.0 | Max gradient magnitude |
| `init_scale` | 0.1 | Weight initialization std |
| `d_model` | 8 | Embedding dimension |
| `epochs` | 300 | Training iterations |

These parameters affect how the model trains. We'll explore their effects in the following parts.



---

## Next: Part 2

You now have a working attention implementation. But is it correct? In Part 2, we'll test it systematically and see if anything surprising happens...